In [1]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# FDA API endpoint
url = "https://api.fda.gov/food/enforcement.json?limit=1000"

response = requests.get(url)
data = response.json()["results"]

# Convert to DataFrame
df = pd.DataFrame(data)

# Convert date
df["recall_initiation_date"] = pd.to_datetime(
    df["recall_initiation_date"],
    format="%Y%m%d",
    errors="coerce"
)

# --------------------------------------------------
# 1. Recall Classification Distribution
# --------------------------------------------------
class_counts = df["classification"].value_counts().reset_index()
class_counts.columns = ["Classification", "Count"]

fig1 = px.bar(
    class_counts,
    x="Classification",
    y="Count",
    title="FDA Food Recall Classifications",
    color="Classification"
)

fig1.show()

# --------------------------------------------------
# 2. Top Recall Reasons
# --------------------------------------------------
reasons = (
    df["reason_for_recall"]
    .value_counts()
    .head(10)
    .reset_index()
)

reasons.columns = ["Reason", "Count"]

fig2 = px.bar(
    reasons,
    x="Count",
    y="Reason",
    orientation="h",
    title="Top 10 Recall Reasons"
)

fig2.show()

# --------------------------------------------------
# 3. Recall Trend Over Time
# --------------------------------------------------
monthly = (
    df.groupby(df["recall_initiation_date"].dt.to_period("M"))
    .size()
    .reset_index(name="Count")
)

monthly["recall_initiation_date"] = monthly["recall_initiation_date"].astype(str)

fig3 = px.line(
    monthly,
    x="recall_initiation_date",
    y="Count",
    title="Food Recall Trend Over Time"
)

fig3.show()

# --------------------------------------------------
# 4. Top Recalling Firms
# --------------------------------------------------
firms = (
    df["recalling_firm"]
    .value_counts()
    .head(10)
    .reset_index()
)

firms.columns = ["Firm", "Count"]

fig4 = px.bar(
    firms,
    x="Count",
    y="Firm",
    orientation="h",
    title="Top 10 Recalling Firms"
)

fig4.show()

#Data_loader.py

In [2]:
import requests
import pandas as pd

def load_fda_data(limit=1000):
    url = f"https://api.fda.gov/food/enforcement.json?limit={limit}"

    response = requests.get(url)
    data = response.json()["results"]

    return pd.DataFrame(data)

#Anaylsis.py

In [3]:
def classify_priority(classification):

    if classification == "Class I":
        return "Critical"

    elif classification == "Class II":
        return "Medium"

    return "Low"

#app.py

In [4]:
import streamlit as st
import pandas as pd
import plotly.express as px

# load_fda_data is defined in the earlier data-loader cell.
# classify_priority is defined in the earlier analysis cell.

st.title("FDA Food Recall Dashboard")

df = load_fda_data()

df["Priority"] = df["classification"].apply(classify_priority)

# Recall Classifications
st.subheader("Recall Classification")

fig = px.bar(
    df["classification"].value_counts().reset_index(),
    x="classification",
    y="count"
)

st.plotly_chart(fig)

# Priority Summary
st.subheader("Priority Distribution")

fig2 = px.pie(
    df,
    names="Priority"
)

st.plotly_chart(fig2)

2026-09-01 20:12:23.521 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:35.604 
  command:

    streamlit run c:\Users\Monic\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-01 20:12:35.607 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:35.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:40.824 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:40.827 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:40.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:

DeltaGenerator()

In [5]:
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Trends",
    "Contamination Types",
    "Geographic Impact",
    "Firm Analysis",
])


2026-09-01 20:12:42.375 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:42.379 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:42.381 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:42.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:42.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:42.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [6]:
import streamlit as st
import pandas as pd
import plotly.express as px
# load_fda_data is defined in the earlier data-loader cell.
from analysis import classify_priority

st.set_page_config(
    page_title="FDA Food Recall Dashboard",
    page_icon="🧪",
    layout="wide",
)

st.title("FDA Food Recall Dashboard")

df = load_fda_data()
if df.empty:
    st.warning("No recall data was returned from the FDA API.")
    st.stop()

df["Priority"] = df["classification"].apply(classify_priority)

# Overview metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Recalls", len(df))
col2.metric("Critical Recalls", int((df["classification"] == "Class I").sum()))
col3.metric("Ongoing Recalls", int((df["status"] == "Ongoing").sum()))
col4.metric("Affected Firms", int(df["recalling_firm"].nunique()))

st.subheader("Recall Classification")
classification_chart = px.pie(df, names="classification", title="Recall Class Distribution")
st.plotly_chart(classification_chart, use_container_width=True)

# Tabs for additional views
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Trends",
    "Contamination Types",
    "Geographic Impact",
    "Firm Analysis",
])


2026-09-01 20:12:44.055 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:44.059 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:44.063 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:44.174 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:48.763 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:48.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:48.769 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:48.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [7]:
with tab1:

    st.header("Dashboard Overview")

    total_recalls = len(df)

    critical = len(
        df[df["classification"] == "Class I"]
    )

    ongoing = len(
        df[df["status"] == "Ongoing"]
    )

    firms = df["recalling_firm"].nunique()

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Total Recalls", total_recalls)
    col2.metric("Critical Recalls", critical)
    col3.metric("Ongoing Recalls", ongoing)
    col4.metric("Affected Firms", firms)

    classification_chart = px.pie(
        df,
        names="classification",
        title="Recall Class Distribution"
    )

    st.plotly_chart(
        classification_chart,
        use_container_width=True
    )

2026-09-01 20:12:56.187 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.211 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.225 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:56.231 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [8]:
with tab2:

    st.header("Recall Trends")

    df["recall_initiation_date"] = pd.to_datetime(
        df["recall_initiation_date"],
        format="%Y%m%d",
        errors="coerce"
    )

    monthly_trend = (
        df.dropna(subset=["recall_initiation_date"])
        .groupby(df["recall_initiation_date"].dt.to_period("M"))
        .size()
        .reset_index(name="Count")
    )

    monthly_trend["Month"] = (
        monthly_trend["recall_initiation_date"]
        .astype(str)
    )

    trend_chart = px.line(
        monthly_trend,
        x="Month",
        y="Count",
        markers=True,
        title="Monthly Recall Trend"
    )

    st.plotly_chart(
        trend_chart,
        use_container_width=True
    )

2026-09-01 20:12:58.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:58.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:58.008 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:58.559 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 20:12:58.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:58.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:58.575 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [9]:
with tab3:

    st.header("Contamination Types")

    top_reasons = (
        df["reason_for_recall"]
        .value_counts()
        .head(15)
        .reset_index()
    )

    top_reasons.columns = [
        "Reason",
        "Count"
    ]

    reason_chart = px.bar(
        top_reasons,
        x="Count",
        y="Reason",
        orientation="h",
        title="Top Recall Causes"
    )

    st.plotly_chart(
        reason_chart,
        use_container_width=True
    )

    st.dataframe(top_reasons)

2026-09-01 20:12:59.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:59.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:59.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:59.916 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 20:12:59.933 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:59.940 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:12:59.948 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# FDA Food Recall Analytics Dashboard

Real-world food recall monitoring project using
the FDA Food Enforcement Dataset.

## Run

streamlit run app.py

In [10]:
with tab4:

    st.header("Geographic Impact")

    state_counts = (
        df["state"]
        .value_counts()
        .head(20)
        .reset_index()
    )

    state_counts.columns = [
        "State",
        "Recalls"
    ]

    geo_chart = px.bar(
        state_counts,
        x="State",
        y="Recalls",
        color="Recalls",
        title="Top States by Recall Count"
    )

    st.plotly_chart(
        geo_chart,
        use_container_width=True
    )

    st.dataframe(state_counts)

2026-09-01 20:13:01.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:01.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:01.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:01.958 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 20:13:01.967 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:01.975 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:01.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [11]:
with tab4:

    st.header("🗺 Geographic Impact")

    # Count recalls per state
    state_counts = (
        df["state"]
        .value_counts()
        .reset_index()
    )

    state_counts.columns = [
        "state",
        "recall_count"
    ]

    # US Choropleth Map
    map_fig = px.choropleth(
        state_counts,
        locations="state",
        locationmode="USA-states",
        color="recall_count",
        scope="usa",
        color_continuous_scale="Reds",
        hover_name="state",
        hover_data=["recall_count"],
        title="Food Recalls by State"
    )

    map_fig.update_layout(
        height=600,
        margin=dict(l=0, r=0, t=50, b=0)
    )

    st.plotly_chart(
        map_fig,
        use_container_width=True
    )

    st.subheader("Top 20 States")

    top_states = (
        state_counts
        .sort_values(
            "recall_count",
            ascending=False
        )
        .head(20)
    )

    st.dataframe(
        top_states,
        use_container_width=True
    )

2026-09-01 20:13:03.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:03.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:03.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:03.459 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 20:13:03.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:03.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:03.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [12]:
selected_states = st.sidebar.multiselect(
    "Select States",
    sorted(df["state"].dropna().unique())
)

if selected_states:
    df = df[df["state"].isin(selected_states)]

2026-09-01 20:13:05.337 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:05.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:05.345 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:05.347 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:05.349 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [13]:
df["Year"] = df["recall_initiation_date"].dt.year
print()

In [14]:
# Sidebar Filters
st.sidebar.header("Filters")

# Year Dropdown
years = sorted(
    df["Year"].dropna().unique(),
    reverse=True
)

selected_year = st.sidebar.selectbox(
    "Select Year",
    options=["All Years"] + list(years)
)

# Apply filter
if selected_year != "All Years":
    df = df[df["Year"] == selected_year]

print()

2026-09-01 20:13:07.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.546 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.548 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.554 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.558 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.558 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:07.560 Session state does not function when running a script without `streamlit run`
2026-09-01 20:13:07.563 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13

In [15]:
selected_years = st.sidebar.multiselect(
    "Select Year(s)",
    options=sorted(df["Year"].dropna().unique()),
    default=sorted(df["Year"].dropna().unique())[-3:]
)

if selected_years:
    df = df[df["Year"].isin(selected_years)]
    
print(df)

2026-09-01 20:13:08.255 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:08.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:08.259 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:08.261 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:08.263 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


         status         city state        country classification openfda  \
14   Terminated   Sioux City    IA  United States        Class I      {}   
17      Ongoing     San Luis    AZ  United States        Class I      {}   
21   Terminated      Detroit    MI  United States       Class II      {}   
25   Terminated  Lake Forest    IL  United States       Class II      {}   
31      Ongoing       Rogers    MN  United States       Class II      {}   
..          ...          ...   ...            ...            ...     ...   
982     Ongoing    Rochester    NY  United States       Class II      {}   
987  Terminated   Brownsburg    IN  United States       Class II      {}   
992  Terminated   Lake Worth    FL  United States        Class I      {}   
994  Terminated      Chicago    IL  United States        Class I      {}   
996  Terminated   Sioux City    IA  United States        Class I      {}   

    product_type event_id              recalling_firm            address_1  \
14       

In [16]:
if selected_year == "All Years":
    st.info("Viewing all available FDA recall records")
else:
    st.info(f"Viewing FDA recall records for {selected_year}")
    print(info)

2026-09-01 20:13:09.974 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:09.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:09.978 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [17]:
st.sidebar.header("Dashboard Filters")

# Year Filter
selected_year = st.sidebar.selectbox(
    "Year",
    options=["All Years"] + sorted(df["Year"].dropna().unique(), reverse=True),
)

if selected_year != "All Years":
    df = df[df["Year"] == selected_year]

# Recall Class Filter
selected_class = st.sidebar.multiselect(
    "Recall Classification",
    options=df["classification"].unique(),
    default=df["classification"].unique()
)

# Priority Filter
selected_priority = st.sidebar.multiselect(
    "Priority",
    options=df["Priority"].unique(),
    default=df["Priority"].unique()
)

# Apply filters
df = df[df["classification"].isin(selected_class)]
df = df[df["Priority"].isin(selected_priority)]

2026-09-01 20:13:10.892 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.894 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.896 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.901 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.903 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.905 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:10.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [18]:
# Product Search
product_search = st.sidebar.text_input(
    "🔍 Search Product Name",
    placeholder="Enter product name..."
)

2026-09-01 20:13:11.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:11.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:11.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:11.961 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:11.963 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:11.964 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [19]:
# Product Name Filter
if product_search:
    df = df[
        df["product_description"]
        .fillna("")
        .str.contains(
            product_search,
            case=False,
            na=False,
            regex=False,
        )
    ]

In [20]:
if product_search:
    matches = len(df)

    st.sidebar.success(
        f"{matches} matching recall(s)"
    )

In [21]:
st.sidebar.header("Dashboard Filters")

# Year
selected_year = st.sidebar.selectbox(
    "Year",
    ["All Years"] + list(years)
)

# Product Search
product_search = st.sidebar.text_input(
    "🔍 Product Search"
)

# Recall Classification
selected_class = st.sidebar.multiselect(
    "Classification",
    options=df["classification"].unique(),
    default=df["classification"].unique()
)

# Priority
selected_priority = st.sidebar.multiselect(
    "Priority",
    options=df["Priority"].unique(),
    default=df["Priority"].unique()
)

2026-09-01 20:13:15.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.479 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:13:15.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [23]:
st.subheader("Critical Recall Search Results")

critical_df = df[df["Priority"] == "Critical"]

st.dataframe(
    critical_df[
        [
            "product_description",
            "recalling_firm",
            "reason_for_recall",
            "status",
        ]
    ],
    use_container_width=True,
)

2026-09-01 20:16:11.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:16:11.927 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:16:11.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:16:11.939 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 20:16:11.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:16:11.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 20:16:11.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()